# 10 · Advanced Recursion

This notebook contains the same set of recursion problems as notebook 07 (the source file is the same), explained from a different angle: **what should the function return?**

The code is exactly my original; I only added explanations and comments.

## The four "answer shapes" of a recursive search

| Question | Function returns | At the base case |
|---|---|---|
| Find **all** answers | nothing (appends to `result`) | `result.append(subset.copy())` |
| Find **any one** answer | `True` / `False` | `return True` when found |
| **Count** the answers | an `int` | `return 1` (valid) or `return 0` (invalid) |
| Find the **best** answer | the best value so far | `return value` |

Decide the shape first, then write the **pick / not-pick** body.

**Recursion tree for `nums = [1,2,3]`** (each level decides one element):
```
                     index 0
            pick 1  /        \  skip 1
                 [1]          []
         pick 2 /   \ skip 2   /   \ ...
            [1,2]   [1]      [2]   []
```
There are `2ⁿ` leaves, one per subset.

## 1. Power set: bitmask
Each number `0 … 2ⁿ−1` is a mask: bit `i` set → include `nums[i]`.

In [ ]:
# power set
nums=[1,2,3]
n=len(nums)
# 1<<n is 2**n
total_subsets=1<<n
result=[]
for num in range(0,total_subsets):
  list=[]
  for i in range(0,n):
    # is bit i set in this mask?
    if num &(1<<i) !=0:
      list.append(nums[i])
  result.append(list)
print(result)


[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]


## 2. Power set: recursion → shape "find all"
Pick, recurse, **backtrack**, then not-pick. Store `subset.copy()` at the base case because `subset` keeps changing.

In [ ]:
def func(index,subset):
  if index >= len(nums):
    # store a COPY, not the list that keeps changing
    result.append(subset.copy())
    return
  # PICK
  subset.append(nums[index])
  func(index+1,subset)
  # BACKTRACK
  subset.pop()
  func(index+1,subset)
nums=[1,2,3]
result=[]
func(0,[])
print(result)


[[1, 2, 3], [1, 2], [1, 3], [1], [2, 3], [2], [3], []]


## 3. Subsequences with sum `k` → shape "find all"
A running `total` is carried down. `total == k` records an answer; `total > k` **prunes** the branch (valid for positive numbers).

In [ ]:
def back_track(index,total,subset):
  # answer found
  if total ==k:
    result.append(subset.copy())
    return
  # prune
  elif total > k:
    return
  if index >=len(nums):
    return
  subset.append(nums[index])
  sum=total+nums[index]
  # explore with the element
  back_track(index+1,sum,subset)
  e=subset.pop()
  sum=sum-e
  # explore without the element
  back_track(index+1,total,subset)
nums=[2,3,6,4,7]
k=7
result=[]
back_track(0,0,[])
print(result)

[[3, 4], [7]]


## 4. One subsequence with sum `k` → shape "find any one"
Return `True` as soon as one answer exists and pass that `True` up through the callers so nothing else is explored.

In [ ]:
def subsequence_with_sum(index,total,subset)->bool:
  if total ==k:
    result.append(subset.copy())
    return True
  elif total > k:
    return False
  if index >=len(nums):
    return False
  subset.append(nums[index])
  sum=total+nums[index]
  pick=subsequence_with_sum(index+1,sum,subset)
  # success → stop exploring
  if pick ==True:
    return True
  subset.pop()
  sum=total
  not_pick =subsequence_with_sum(index+1,total,subset)
  # otherwise the not-pick branch decides
  return not_pick
nums=[2,3,6,4,7]
k=11
result=[]
print(subsequence_with_sum(0,0,[]))



True


## 5. Count subsequences with sum `k` → shape "count"
`pick + not_pick`. Many `(index, total)` states repeat, which is exactly what **dynamic programming** (memoisation) later removes.

In [ ]:
def no_of_subsequence_with_sum_k(index,total):
  if total ==k:

    return 1
  elif total > k:
    return 0
  if index >=len(nums):
    return 0

  sum=total+nums[index]
  pick=no_of_subsequence_with_sum_k(index+1,sum)

  sum=total
  not_pick=no_of_subsequence_with_sum_k(index+1,sum)
  # ways with the element + ways without it
  return pick +not_pick
nums=[2,3,6,4,7]
k=9
print(no_of_subsequence_with_sum_k(0,0))

3


## 6. Binary strings with no consecutive 1s
The problem statement is in the next cell (a comment). ⚠️ The output `{1, 2, 4, 6}` shown under it is a **stale output** from an earlier cell.

In [ ]:
#generate all the binary string
#Given an integer N , Print all binary strings of size N which do not contain consecutive 1s.

#A binary string is that string which contains only 0 and 1.



{1, 2, 4, 6}


`flag` tells whether a `1` may be placed (it can't if the previous character was `1`). `0` is always allowed. After trying `1`, set the position back to `0` (**backtrack**).

In [ ]:
class solution:
  def solve(self,index,flag,numbers,result):
    if index >=len(numbers):
      result.append(''.join(numbers))
      return
    # placing 0 is always allowed
    numbers[index]='0'
    self.solve(index+1,True,numbers,result)
    # placing 1 allowed only after a 0
    if flag ==True:
      numbers[index]="1"
      self.solve(index+1,False,numbers,result)
      # backtrack
      numbers[index]="0"
  def generate_binary_strings(self,n):
    numbers=["0"]*n
    result=[]
    self.solve(0,True,numbers,result)
    return result
s=solution()
print(s.generate_binary_strings(4))

['0000', '0001', '0010', '0100', '0101', '1000', '1001', '1010']


---
### ✅ Key takeaways
- First decide the **answer shape**, then write the pick / not-pick body.
- `subset.copy()` when storing, `subset.pop()` to backtrack.
- Prune early when the problem allows it.